# Hadoop MapReduce with Python
There are two prominent *Python* APIs for interfacing *Hadoop MapReduce* clusters:

## *Snakebite* for *HDFS* access
The [Snakebite Lib](https://github.com/spotify/snakebite) allows easy access to *HDFS* file systems:  
```
>>> from snakebite.client import Client
>>> client = Client("localhost", 8020, use_trash=False)
>>> for x in client.ls(['/']):
...     print x
```

See [documentation](https://snakebite.readthedocs.io/en/latest/) for details.


## *MRJOB* for *MapReduce* job execution
The ``mrjob`` lib -> [see docu](https://mrjob.readthedocs.io/en/latest/index.html) is a power full *MapReduce* client for *Python*. Some of the key features are:

* local emulation (single and multi-core) a *Hadoop* cluster for development and debugging
* simple access, authentication and file transfer to *Hadoop* clusters
* powerful API for common cloud services, such as AWS or Azure   

In [1]:
#in colab, we need to clone the data from the repo
!git clone https://github.com/keuperj/DATA.git

Cloning into 'DATA'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 126 (delta 11), reused 39 (delta 11), pack-reused 87 (from 1)
Receiving objects: 100% (126/126), 185.56 MiB | 16.45 MiB/s, done.
Resolving deltas: 100% (32/32), done.
Updating files: 100% (86/86), done.


### Preparing our environment

In [2]:
!pip install mrjob boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.6/439.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.6 MB/s eta 0:00:00


## A *MRJOB* Example: WordCount (again)
Since *Hadoop* works only on file in- and outputs, we do not have usual function based API. We need to pass our code (implementation of *Map* and *Reduce*) as executable *Python* scripts:

* use *Jupyter's* ``%%file`` magic command to write the cell to file
* create a executable script with ``__main__`` method
* inherit from the ``MRJob`` class
* implement ``mapper()`` and ``reducer()`` methods
* call ``run()`` at start

In [3]:
%%file wordcount.py
#this will save this cell as file

from mrjob.job import MRJob

class MRWordCount(MRJob):
    def mapper(self, _, line):
        for word in line.split():
            yield(word, 1)

    def reducer(self, word, counts):
        yield(word, sum(counts))

if __name__ == '__main__':
    MRWordCount.run()


Writing wordcount.py


### execute script from cmd
* ``-r local`` causes local multi-core emulation a *Hadoop* cluster.
* Input files are cmd arguments
* define ouput-file (see docs) or use streams: `` > out.txt``

In [4]:
! python wordcount.py -r local DATA/text1.rst DATA/text2.rst DATA/text3.rst

No configs found; falling back on auto-configuration
No configs specified for local runner
Creating temp directory /tmp/wordcount.root.20260410.124559.804096
Running step 1 of 1...
job output is in /tmp/wordcount.root.20260410.124559.804096/output
Streaming final output from /tmp/wordcount.root.20260410.124559.804096/output...
"-"	3
"--"	6
"A"	16
"Adjusting"	3
"Alex"	3
"All"	3
"Almighty,"	3
"Alphabet"	3
"Amazingly"	3
"And"	3
"Bawds"	3
"Baz,"	3
"Big"	6
"Blind"	9
"Blowzy"	3
"Bookmarksgrove"	4
"Bookmarksgrove,"	3
"Brawny"	3
"Brick"	3
"Bright"	3
"But"	3
"Commas,"	3
"Consonantia,"	4
"Copy"	3
"Cozy"	3
"Crazy"	3
"DJs"	4
"Duden"	4
"Even"	3
"Far"	2
"Few"	3
"Five"	3
"Flummoxed"	3
"Fox"	3
"Foxy"	3
"Fredericka"	3
"God!"	3
"Grammar."	3
"Have"	3
"How"	3
"I"	44
"Ipsum"	3
"Iraq."	3
"It"	4
"Italic"	3
"Jack!\""	3
"Jack"	3
"Japan"	3
"Jim."	6
"Joaquin"	3
"July"	3
"Junk"	3
"Lane."	3
"Line"	3
"Little"	9
"Longe"	3
"Lorem"	3
"MTV"	9
"Marks"	3
"Mountains,"	3
"My"	6
"O"	3
"Oh,"	3
"On"	3
"One"	3
"Oxmox"	3
"Parol

## Execution on AWS EMR
AWS EMR is a clound formation service which allows you to create *Hadoop*, *Spark* and other data analytics clusters with a few clicks.

**NOTE**: we are not endorsing AWS specifically, other cloud service providers have similar offers



### Case 1: create cluster on the fly
We create a cluster just for a single job:
* simple solution for large jobs that run only once (or only at sparse points in time)
* this approach cause a lot of over head: not suitable for small and frequent jobs  

First, we need a config file for the connection to EMR:
**fill in YOUR AWS credentials**

In [17]:
%%file mrjob.conf
runners:
  emr:
    aws_access_key_id: c
    aws_secret_access_key: c
    instance_type: m5.xlarge
    num_core_instances: 2
    region: eu-west-1

Overwriting mrjob.conf


In [18]:
! python wordcount.py -r emr --bootstrap-mrjob DATA/text1.rst DATA/text2.rst -c mrjob.conf


Traceback (most recent call last):
  File "/content/wordcount.py", line 14, in <module>
    MRWordCount.run()
  File "/usr/local/lib/python3.12/dist-packages/mrjob/job.py", line 616, in run
    cls().execute()
  File "/usr/local/lib/python3.12/dist-packages/mrjob/job.py", line 687, in execute
    self.run_job()
  File "/usr/local/lib/python3.12/dist-packages/mrjob/job.py", line 634, in run_job
    with self.make_runner() as runner:
         ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/mrjob/job.py", line 713, in make_runner
    return self._runner_class()(**self._runner_kwargs())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/mrjob/emr.py", line 358, in __init__
    self._fix_s3_tmp_and_log_uri_opts()
  File "/usr/local/lib/python3.12/dist-packages/mrjob/emr.py", line 599, in _fix_s3_tmp_and_log_uri_opts
    self._set_cloud_tmp_dir()
  File "/usr/local/lib/python3.12/dist-packages/mrjob/emr.py", line 617, i

### Case 2: connect to existing cluster

In [19]:
%%file mrjob_cluster.conf
runners:
  emr:
    aws_access_key_id: cc
    aws_secret_access_key: cc
    region: eu-west-1

Overwriting mrjob_cluster.conf


We need the **ID** of the cluster we want to connect to.

In [20]:
! python wordcount.py -r emr --cluster-id=j-cluster DATA/text1.rst DATA/text2.rst -c mrjob_cluster.conf

Using s3://mrjob-757042689da50525/tmp/ as our temp dir on S3
Creating temp directory /tmp/wordcount.root.20260410.131809.649225
uploading working dir files to s3://mrjob-757042689da50525/tmp/wordcount.root.20260410.131809.649225/files/wd...
Copying other local files to s3://mrjob-757042689da50525/tmp/wordcount.root.20260410.131809.649225/files/
Adding our job to existing cluster j-250SRDSHW5FGD
  master node is ec2-3-250-176-155.eu-west-1.compute.amazonaws.com
Waiting for Step 1 of 1 (s-0976963M1DJXPDEE5SY) to complete...
  RUNNING for 0:00:28
  COMPLETED
Attempting to fetch counters from logs...
Waiting 10 minutes for logs to transfer to S3... (ctrl-c to skip)

To fetch logs immediately next time, set up SSH. See:
https://pythonhosted.org/mrjob/guides/emr-quickstart.html#configuring-ssh-credentials

Looking for step log in s3://aws-logs-846657657993-eu-west-1/elasticmapreduce/j-250SRDSHW5FGD/steps/s-0976963M1DJXPDEE5SY...
  Parsing step log: s3://aws-logs-846657657993-eu-west-1/elasti

## Exercise
Use  *mrjob*  to  compute  employee  **top  annual  salaries** and  **gross pay** in the *CSV* table ``Baltimore_City_employee_Salaries_FY2014.csv``.

* use  ``import csv`` to read the data -> [API docs](https://docs.python.org/3/library/csv.html)
* use ``yield`` to return *producers* from *map* and *reduce* functions
* return top entries in both categories

In [24]:
%%writefile top_salaries_mrjob.py
import csv
from io import StringIO
from mrjob.job import MRJob

class MRSalaries(MRJob):
    def configure_args(self):
        super().configure_args()
        self.add_passthru_arg(
            "--top-n",
            type=int,
            default=10,
            help="Number of top entries to return for AnnualSalary and GrossPay",
        )

    def mapper(self, _, line):
        if not line.strip():
            return

        if line.startswith("Name,JobTitle,AgencyID,Agency,HireDate,AnnualSalary,GrossPay"):
            return

        row = next(csv.reader(StringIO(line)))
        if len(row) < 7:
            return

        name = row[0].strip()
        job_title = row[1].strip()

        try:
            annual_salary = float(row[5])
            gross_pay = float(row[6])
        except ValueError:
            return

        yield "AnnualSalary", (annual_salary, name, job_title)
        yield "GrossPay", (gross_pay, name, job_title)

    def reducer(self, category, values):
        top_n = self.options.top_n
        sorted_values = sorted(values, key=lambda item: item[0], reverse=True)[:top_n]

        for rank, (amount, name, job_title) in enumerate(sorted_values, start=1):
            yield category, {
                "rank": rank,
                "name": name,
                "job_title": job_title,
                "amount": round(amount, 2),
            }
if __name__ == "__main__":
    MRSalaries.run()

Overwriting top_salaries_mrjob.py


In [25]:
!python top_salaries_mrjob.py DATA/Baltimore_City_Employee_Salaries_FY2014.csv -r local --no-conf --top-n 10

No configs specified for local runner
Creating temp directory /tmp/top_salaries_mrjob.root.20260410.142424.048942
Running step 1 of 1...
job output is in /tmp/top_salaries_mrjob.root.20260410.142424.048942/output
Streaming final output from /tmp/top_salaries_mrjob.root.20260410.142424.048942/output...
"AnnualSalary"	{"rank": 1, "name": "Bernstein,Gregg L", "job_title": "STATE'S ATTORNEY", "amount": 238772.0}
"AnnualSalary"	{"rank": 2, "name": "Charles,Ronnie E", "job_title": "EXECUTIVE LEVEL III", "amount": 200000.0}
"AnnualSalary"	{"rank": 3, "name": "Batts,Anthony W", "job_title": "EXECUTIVE LEVEL III", "amount": 193800.0}
"AnnualSalary"	{"rank": 4, "name": "Black,Harry E", "job_title": "EXECUTIVE LEVEL III", "amount": 190000.0}
"AnnualSalary"	{"rank": 5, "name": "Swift,Michael", "job_title": "CONTRACT SERV SPEC II", "amount": 187200.0}
"AnnualSalary"	{"rank": 6, "name": "Parthemos,Kaliope", "job_title": "EXECUTIVE LEVEL III", "amount": 172000.0}
"AnnualSalary"	{"rank": 7, "name": "F

In [26]:
!python top_salaries_mrjob.py -r emr --cluster-id=j-cluster DATA/Baltimore_City_Employee_Salaries_FY2014.csv -c mrjob_cluster.conf --top-n 10

Using s3://mrjob-757042689da50525/tmp/ as our temp dir on S3
Creating temp directory /tmp/top_salaries_mrjob.root.20260410.142435.640757
uploading working dir files to s3://mrjob-757042689da50525/tmp/top_salaries_mrjob.root.20260410.142435.640757/files/wd...
Copying other local files to s3://mrjob-757042689da50525/tmp/top_salaries_mrjob.root.20260410.142435.640757/files/
Adding our job to existing cluster j-250SRDSHW5FGD
  master node is ec2-3-250-176-155.eu-west-1.compute.amazonaws.com
Waiting for Step 1 of 1 (s-05201393P7EL2COZI6P7) to complete...
  RUNNING for 0:00:24
  COMPLETED
Attempting to fetch counters from logs...
Waiting 10 minutes for logs to transfer to S3... (ctrl-c to skip)

To fetch logs immediately next time, set up SSH. See:
https://pythonhosted.org/mrjob/guides/emr-quickstart.html#configuring-ssh-credentials

Looking for step log in s3://aws-logs-846657657993-eu-west-1/elasticmapreduce/j-250SRDSHW5FGD/steps/s-05201393P7EL2COZI6P7...
  Parsing step log: s3://aws-logs-